# Runtime Analysis

---
## 1. Imports and Configuration

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

EVAL_DIR = Path(".")

FILENAME_RE = re.compile(r"evaluation_(FB|NELL)_(\d+)_(\w+)_(\w+)_random\.csv")

DATASET_NAME_MAP = {"FB": "FB15k-237+H", "NELL": "NELL995+H"}
DATASET_ORDER    = ["FB15k-237+H", "NELL995+H"]

MODEL_ORDER   = ["cqd", "betae", "query2box", "gqe"]
MODEL_DISPLAY = {"cqd": "CQD", "betae": "BetaE", "query2box": "Query2Box", "gqe": "GQE"}

QUERY_TYPE_ORDER = ["2p", "3p", "2i", "3i", "2u", "up", "ip", "pi"]
QTYPE_DISPLAY = {
    "2p": "2p",  "3p": "3p",  "2i": "2i",  "3i": "3i",
    "2u": "2u",  "up": "2u1p", "ip": "1p2i", "pi": "2i1p",
}

# K values to sweep for the runtime-vs-K plot
K_SWEEP = [1, 3, 5, 10]

DRAWIO = {
    "blue":   {"fill": "#B0E3E6", "line": "#0E8088"},
    "orange": {"fill": "#FAD7AC", "line": "#B46504"},
    "purple": {"fill": "#D0CEE2", "line": "#56517E"},
    "red":    {"fill": "#FAD9D5", "line": "#AE4132"},
}
MODEL_COLORS = {"betae": DRAWIO["blue"], "cqd": DRAWIO["orange"],
                 "gqe": DRAWIO["purple"], "query2box": DRAWIO["red"]}
FONT_FAMILY = "Times New Roman, Times, serif"

FIGURES_DIR = EVAL_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

## 2. Discover and Load Files

In [2]:
records = []
for path in sorted(EVAL_DIR.glob("evaluation_*_random.csv")):
    m = FILENAME_RE.match(path.name)
    if not m:
        continue
    dataset_raw, _version, model, qtype = m.groups()
    dataset = DATASET_NAME_MAP.get(dataset_raw, dataset_raw)

    df = pd.read_csv(path, usecols=["type", "iteration", "time"])
    avg_ms = df["time"].mean() * 1000.0
    records.append({
        "dataset": dataset, "model": model, "qtype": qtype,
        "avg_ms": avg_ms, "n_iterations": df["iteration"].nunique(),
    })

df_all = pd.DataFrame(records)
print(f"Loaded {len(records)} files")
print("Datasets:", sorted(df_all["dataset"].unique()))
print("Query types:", sorted(df_all["qtype"].unique()))
print("Models:", sorted(df_all["model"].unique()))
print("\nIterations stored per file (sanity check):")
print(df_all["n_iterations"].value_counts())

# Which (dataset, model, qtype) combinations are missing -- shown explicitly
# instead of silently absent from the table, in case a future run is partial
# again (as the NELL/CQD run was earlier in this project).
expected = pd.MultiIndex.from_product(
    [DATASET_ORDER, MODEL_ORDER, QUERY_TYPE_ORDER], names=["dataset", "model", "qtype"]
)
present = pd.MultiIndex.from_frame(df_all[["dataset", "model", "qtype"]])
missing = expected.difference(present)
if len(missing):
    print(f"\n{len(missing)} missing (dataset, model, qtype) combinations:")
    for row in missing:
        print(" ", row)
else:
    print("\nNo missing (dataset, model, qtype) combinations.")

Loaded 64 files
Datasets: ['FB15k-237+H', 'NELL995+H']
Query types: ['2i', '2p', '2u', '3i', '3p', 'ip', 'pi', 'up']
Models: ['betae', 'cqd', 'gqe', 'query2box']

Iterations stored per file (sanity check):
n_iterations
10    64
Name: count, dtype: int64

No missing (dataset, model, qtype) combinations.


## 3. Main Runtime Table (Table~\ref{tab:runtimes})

In [3]:
def fmt(val):
    return f"{val:.2f}" if pd.notna(val) else "--"


col_order = [q for q in QUERY_TYPE_ORDER if q in df_all["qtype"].unique()]
col_headers = [QTYPE_DISPLAY[q] for q in col_order]

pivot = df_all.groupby(["dataset", "model", "qtype"])["avg_ms"].mean().unstack("qtype").reindex(columns=col_order)
dataset_avg = df_all.groupby(["dataset", "qtype"])["avg_ms"].mean().unstack("qtype").reindex(columns=col_order)
overall_avg = df_all.groupby("qtype")["avg_ms"].mean().reindex(col_order)

dataset_order = [d for d in DATASET_ORDER if d in df_all["dataset"].unique()]
model_order   = [m for m in MODEL_ORDER if m in df_all["model"].unique()]

n_cols = 2 + len(col_order)  # Dataset + Model + query-type columns
cmidrule = f"\\cmidrule(l){{2-{n_cols}}}"
avg_label, overall_label, empty = r"\textit{Average}", r"\textit{Overall Average}", ""

rows_latex = []
for i, dataset in enumerate(dataset_order):
    if i > 0:
        rows_latex.append(r"    \midrule")
    first = True
    for model in model_order:
        if (dataset, model) not in pivot.index:
            continue
        vals = " & ".join(fmt(pivot.loc[(dataset, model), c]) for c in col_order)
        dataset_cell = dataset if first else empty
        rows_latex.append(f"        {dataset_cell:<20} & {MODEL_DISPLAY[model]:<20} & {vals} \\\\")
        first = False
    avg_vals = " & ".join(fmt(dataset_avg.loc[dataset, c]) for c in col_order)
    rows_latex.append(f"    {cmidrule}")
    rows_latex.append(f"        {empty:<20} & {avg_label:<20} & {avg_vals} \\\\")

rows_latex.append(r"    \midrule")
overall_vals = " & ".join(fmt(overall_avg[c]) for c in col_order)
rows_latex.append(f"    \\multicolumn{{2}}{{@{{}}l@{{}}}}{{{overall_label}}} & {overall_vals} \\\\")

col_headers_latex = " & ".join(f"\\textbf{{{c}}}" for c in col_headers)
col_fmt = "@{}ll@{\\hspace{5pt}}" + "r" * len(col_order) + "@{}"

latex = r"""\begin{table}[t]
\caption{Average runtime (in milliseconds) for computing Shapley values per query--answer pair, reported by query type and model on each benchmark.}
\label{tab:runtimes}
\footnotesize
\centering
\setlength{\tabcolsep}{3pt}
\begin{tabular*}{\columnwidth}{""" + col_fmt + "}\n"
latex += r"    \toprule" + "\n"
latex += f"    \\textbf{{Dataset}} & \\textbf{{Model}} & {col_headers_latex} \\\\\n"
latex += r"    \midrule" + "\n"
latex += "\n".join(rows_latex) + "\n"
latex += r"    \bottomrule" + "\n"
latex += r"\end{tabular*}" + "\n"
latex += r"\end{table}"

print(latex)

with open("table_runtime.tex", "w") as f:
    f.write(latex + "\n")
print("\nTable saved to evaluations/table_runtime.tex")

\begin{table}[t]
\caption{Average runtime (in milliseconds) for computing Shapley values per query--answer pair, reported by query type and model on each benchmark.}
\label{tab:runtimes}
\footnotesize
\centering
\setlength{\tabcolsep}{3pt}
\begin{tabular*}{\columnwidth}{@{}ll@{\hspace{5pt}}rrrrrrrr@{}}
    \toprule
    \textbf{Dataset} & \textbf{Model} & \textbf{2p} & \textbf{3p} & \textbf{2i} & \textbf{3i} & \textbf{2u} & \textbf{2u1p} & \textbf{1p2i} & \textbf{2i1p} \\
    \midrule
        FB15k-237+H          & CQD                  & 5.09 & 15.94 & 6.83 & 17.51 & 10.79 & 13.99 & 14.33 & 13.76 \\
                             & BetaE                & 5.03 & 13.30 & 7.86 & 17.84 & 13.53 & 13.93 & 13.52 & 13.01 \\
                             & Query2Box            & 4.79 & 12.82 & 5.93 & 14.02 & 8.45 & 14.02 & 13.59 & 12.71 \\
                             & GQE                  & 4.80 & 12.82 & 5.67 & 13.42 & 7.93 & 14.75 & 15.60 & 13.05 \\
    \cmidrule(l){2-10}
                      